# Untrained PyTorch MLP for motor-imagery classification

This notebook **only creates the neural-network architecture**. It does not train, validate, test, make predictions, calculate accuracy, or open the test split.

The MLP is designed to eventually classify left- versus right-hand motor imagery from 54 compact EEG features:

- 27 mean mu-band features, one for each scalp electrode; and
- 27 mean beta-band features, one for each scalp electrode.

These are the same feature definitions used by the logistic-regression sanity check. The intended representation is the full 0–5 second post-cue window, although no trial data are loaded here.

## 1. Import PyTorch and configure reproducible initialization

PyTorch randomly initializes the network's weights. Setting a seed ensures that rerunning the notebook creates the same initial weights. This is not model training; it only makes architecture creation reproducible.

In [ ]:
from pathlib import Path
import copy
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cpu')
print('PyTorch version:', torch.__version__)
print('Initialization device:', DEVICE)
print('Initialization seed:', SEED)

## 2. Define the 54 expected input features

No EEG trials are loaded in this cell. It only defines the feature names and their required order.

For each electrode:

- `*_mu_mean_z` represents mean standardized mu activity from 8 to <13 Hz; and
- `*_beta_mean_z` represents mean standardized beta activity from 13 to 30 Hz.

Participant identity, dataset, run, phase, performance, gender, file path, and trial label are not model inputs.

In [ ]:
ELECTRODES = [
    'Fz', 'FCz', 'Cz', 'CPz', 'Pz',
    'C1', 'C3', 'C5', 'C2', 'C4', 'C6',
    'F4', 'FC2', 'FC4', 'FC6', 'CP2', 'CP4', 'CP6', 'P4',
    'F3', 'FC1', 'FC3', 'FC5', 'CP1', 'CP3', 'CP5', 'P3',
]
FEATURE_NAMES = (
    [f'{electrode}_mu_mean_z' for electrode in ELECTRODES]
    + [f'{electrode}_beta_mean_z' for electrode in ELECTRODES]
)

assert len(ELECTRODES) == 27
assert len(FEATURE_NAMES) == 54

feature_table = pd.DataFrame({
    'input_position': range(1, 55),
    'feature_name': FEATURE_NAMES,
    'electrode': ELECTRODES + ELECTRODES,
    'band': ['mu'] * 27 + ['beta'] * 27,
    'frequency_range_hz': ['8 to <13'] * 27 + ['13 to 30'] * 27,
})
display(feature_table)
print('Number of expected MLP inputs:', len(FEATURE_NAMES))

## 3. Define the MLP architecture

The architecture is `54 → 64 → 32 → 2`:

1. **Input layer: 54 values.** These are the 27 mu and 27 beta electrode features.
2. **First dense layer: 64 hidden units.** Each unit can combine information across electrodes and bands, such as contrasting C3 mu activity with C4 mu activity.
3. **Batch normalization.** This is designed to stabilize the 64 hidden activations during future training.
4. **ReLU activation.** ReLU allows the network to represent nonlinear interactions instead of acting like logistic regression.
5. **30% dropout.** During future training, this would randomly hide 30% of first-layer activations to reduce overdependence on individual electrodes.
6. **Second dense layer: 32 hidden units.** This compresses the first-layer patterns into a smaller representation.
7. **Second ReLU and 20% dropout.** These add another nonlinear stage and regularization.
8. **Output layer: 2 logits.** Output position 0 represents left and position 1 represents right. They are raw scores, not probabilities.

Defining these layers creates trainable parameters, but none of those parameters are updated in this notebook.

In [ ]:
class MotorImageryMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=54, out_features=64),
            nn.BatchNorm1d(num_features=64),
            nn.ReLU(),
            nn.Dropout(p=0.30),
            nn.Linear(in_features=64, out_features=32),
            nn.ReLU(),
            nn.Dropout(p=0.20),
            nn.Linear(in_features=32, out_features=2),
        )

    def forward(self, inputs):
        return self.network(inputs)

model = MotorImageryMLP().to(DEVICE)
print(model)

## 4. Describe the untrained parameters

This cell reports how many adjustable numbers the architecture contains. It does not pass data through the network and does not modify any weight.

A small parameter count is intentional: the compact 54-feature representation does not justify a very large dense network.

In [ ]:
parameter_rows = []
for parameter_name, parameter in model.named_parameters():
    parameter_rows.append({
        'parameter': parameter_name,
        'shape': tuple(parameter.shape),
        'values': parameter.numel(),
        'trainable': parameter.requires_grad,
    })

parameter_table = pd.DataFrame(parameter_rows)
display(parameter_table)
print(f'Total trainable parameters: {parameter_table.loc[parameter_table["trainable"], "values"].sum():,}')
print('Training performed: NO')
print('Testing performed:  NO')

## 5. Optionally save the untrained architecture state

Saving the initial state makes the exact random initialization reproducible. The filename and checkpoint metadata explicitly identify it as untrained so it cannot be mistaken for a fitted classifier.

This cell does not create an optimizer, calculate a loss, make a prediction, or evaluate any data.

In [ ]:
def find_project_data(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return parent / 'data'
    raise FileNotFoundError('Could not locate the project data directory.')

DATA_ROOT = find_project_data()
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'neural_net'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
UNTRAINED_MODEL_PATH = OUTPUT_ROOT / 'untrained_motor_imagery_mlp.pt'
CONFIGURATION_PATH = OUTPUT_ROOT / 'untrained_motor_imagery_mlp.json'

checkpoint = {
    'model_state_dict': model.state_dict(),
    'architecture': '54 -> 64 -> 32 -> 2',
    'feature_names': FEATURE_NAMES,
    'class_names': ['left', 'right'],
    'seed': SEED,
    'trained': False,
    'tested': False,
}
torch.save(checkpoint, UNTRAINED_MODEL_PATH)

configuration = {key: value for key, value in checkpoint.items() if key != 'model_state_dict'}
with open(CONFIGURATION_PATH, 'w', encoding='utf-8') as destination:
    json.dump(configuration, destination, indent=2)

print(f'Untrained checkpoint: {UNTRAINED_MODEL_PATH}')
print(f'Configuration:       {CONFIGURATION_PATH}')
print('The network has not been trained or tested.')

## 6. Train the model on training participants and monitor validation participants

This is the only cell that trains the MLP. It loads the previously prepared full post-cue inputs, standardizes the 54 compact features using training trials only, and updates the network with mini-batch gradient descent.

The optimizer is AdamW with a small weight-decay penalty. Cross-entropy loss measures how strongly the predicted left/right scores disagree with the true label. Training may run for at most 100 epochs, but it stops early when validation loss fails to improve for 12 epochs. The weights from the lowest validation-loss epoch are restored.

Reported metrics include:

- cross-entropy loss;
- accuracy;
- balanced accuracy, which averages left and right recall;
- macro F1;
- ROC AUC;
- Cohen's kappa; and
- the four confusion-matrix counts.

No test participant is loaded or evaluated in this cell.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, f1_score, roc_auc_score,
)

# Load only the training and validation inputs created by the sanity-check notebook.
TRAINING_INPUT_PATH = (
    DATA_ROOT / 'processed' / 'sanity_check_classifier' / 'inputs'
    / 'full_post_cue_all_model_inputs.csv'
)
training_inputs = pd.read_csv(TRAINING_INPUT_PATH)
assert set(training_inputs['split']) == {'train', 'validation'}
assert 'test' not in set(training_inputs['split'])
assert training_inputs.loc[training_inputs['split'].eq('train'), 'participant'].nunique() == 55
assert training_inputs.loc[training_inputs['split'].eq('validation'), 'participant'].nunique() == 12

train_frame = training_inputs.loc[training_inputs['split'].eq('train')].reset_index(drop=True)
validation_frame = training_inputs.loc[training_inputs['split'].eq('validation')].reset_index(drop=True)
X_train_raw = train_frame[FEATURE_NAMES].to_numpy(dtype=np.float32)
X_validation_raw = validation_frame[FEATURE_NAMES].to_numpy(dtype=np.float32)
y_train_array = train_frame['label_id'].to_numpy(dtype=np.int64)
y_validation_array = validation_frame['label_id'].to_numpy(dtype=np.int64)

# This second scaling step is fitted only to the 54 aggregated training features.
feature_mean = X_train_raw.mean(axis=0, dtype=np.float64).astype(np.float32)
feature_std = X_train_raw.std(axis=0, dtype=np.float64).astype(np.float32)
assert np.all(feature_std > 0)
X_train_array = np.ascontiguousarray((X_train_raw - feature_mean) / feature_std, dtype=np.float32)
X_validation_array = np.ascontiguousarray(
    (X_validation_raw - feature_mean) / feature_std, dtype=np.float32
)

# The installed Torch build cannot use torch.from_numpy with NumPy 2.x.
# frombuffer preserves the numeric values without invoking that bridge.
X_train = torch.frombuffer(memoryview(X_train_array), dtype=torch.float32).reshape(X_train_array.shape)
X_validation = torch.frombuffer(
    memoryview(X_validation_array), dtype=torch.float32
).reshape(X_validation_array.shape)
y_train_array = np.ascontiguousarray(y_train_array)
y_validation_array = np.ascontiguousarray(y_validation_array)
y_train = torch.frombuffer(memoryview(y_train_array), dtype=torch.int64).reshape(y_train_array.shape)
y_validation = torch.frombuffer(
    memoryview(y_validation_array), dtype=torch.int64
).reshape(y_validation_array.shape)

BATCH_SIZE = 128
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    TensorDataset(X_train, y_train), batch_size=BATCH_SIZE,
    shuffle=True, generator=generator,
)
validation_loader = DataLoader(
    TensorDataset(X_validation, y_validation), batch_size=BATCH_SIZE, shuffle=False
)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
MAX_EPOCHS = 100
PATIENCE = 12
MIN_IMPROVEMENT = 1e-4

def run_training_epoch():
    model.train()
    loss_total = 0.0
    examples = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_X.to(DEVICE))
        loss = loss_function(logits, batch_y.to(DEVICE))
        loss.backward()
        optimizer.step()
        loss_total += loss.item() * len(batch_X)
        examples += len(batch_X)
    return loss_total / examples

@torch.no_grad()
def evaluate_loader(loader):
    model.eval()
    loss_total = 0.0
    examples = 0
    targets, predictions, right_probabilities = [], [], []
    for batch_X, batch_y in loader:
        logits = model(batch_X.to(DEVICE))
        loss = loss_function(logits, batch_y.to(DEVICE))
        probabilities = torch.softmax(logits, dim=1)
        predicted = logits.argmax(dim=1)
        loss_total += loss.item() * len(batch_X)
        examples += len(batch_X)
        targets.extend(batch_y.tolist())
        predictions.extend(predicted.tolist())
        right_probabilities.extend(probabilities[:, 1].tolist())
    targets = np.asarray(targets, dtype=int)
    predictions = np.asarray(predictions, dtype=int)
    right_probabilities = np.asarray(right_probabilities, dtype=float)
    matrix = confusion_matrix(targets, predictions, labels=[0, 1])
    return {
        'loss': loss_total / examples,
        'accuracy': accuracy_score(targets, predictions),
        'balanced_accuracy': balanced_accuracy_score(targets, predictions),
        'macro_f1': f1_score(targets, predictions, average='macro'),
        'roc_auc': roc_auc_score(targets, right_probabilities),
        'cohen_kappa': cohen_kappa_score(targets, predictions),
        'true_left_pred_left': int(matrix[0, 0]),
        'true_left_pred_right': int(matrix[0, 1]),
        'true_right_pred_left': int(matrix[1, 0]),
        'true_right_pred_right': int(matrix[1, 1]),
        'targets': targets,
        'predictions': predictions,
        'right_probability': right_probabilities,
    }

# Reset initialization so rerunning this cell produces the same training run.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
model = MotorImageryMLP().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

history_rows = []
best_state = None
best_epoch = 0
best_validation_loss = np.inf
epochs_without_improvement = 0
started = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    training_loss = run_training_epoch()
    validation_epoch = evaluate_loader(validation_loader)
    history_rows.append({
        'epoch': epoch,
        'training_loss': training_loss,
        'validation_loss': validation_epoch['loss'],
        'validation_balanced_accuracy': validation_epoch['balanced_accuracy'],
        'validation_macro_f1': validation_epoch['macro_f1'],
        'validation_roc_auc': validation_epoch['roc_auc'],
    })

    if validation_epoch['loss'] < best_validation_loss - MIN_IMPROVEMENT:
        best_validation_loss = validation_epoch['loss']
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0 or epochs_without_improvement >= PATIENCE:
        print(
            f'Epoch {epoch:>3}: train loss={training_loss:.4f}; '
            f'validation loss={validation_epoch["loss"]:.4f}; '
            f'validation balanced accuracy={validation_epoch["balanced_accuracy"]:.3f}'
        )
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

model.load_state_dict(best_state)
training_result = evaluate_loader(train_loader)
validation_result = evaluate_loader(validation_loader)

performance_columns = [
    'loss', 'accuracy', 'balanced_accuracy', 'macro_f1', 'roc_auc', 'cohen_kappa',
    'true_left_pred_left', 'true_left_pred_right',
    'true_right_pred_left', 'true_right_pred_right',
]
performance = pd.DataFrame([
    {'split': 'train', 'participants': 55, 'trials': len(train_frame),
     **{column: training_result[column] for column in performance_columns}},
    {'split': 'validation', 'participants': 12, 'trials': len(validation_frame),
     **{column: validation_result[column] for column in performance_columns}},
])
display(performance)

history = pd.DataFrame(history_rows)
history.to_csv(OUTPUT_ROOT / 'training_history.csv', index=False)
performance.to_csv(OUTPUT_ROOT / 'training_validation_metrics.csv', index=False)
np.savez_compressed(
    OUTPUT_ROOT / 'trained_feature_scaler.npz',
    feature_names=np.asarray(FEATURE_NAMES), mean=feature_mean, std=feature_std,
    training_participants=np.asarray(sorted(set(train_frame['participant']))),
)

validation_predictions = validation_frame[[
    'sample_id', 'dataset', 'participant', 'run', 'phase',
    'trial', 'class_label', 'label_id',
]].copy()
validation_predictions['predicted_label_id'] = validation_result['predictions']
validation_predictions['predicted_class'] = np.where(
    validation_result['predictions'] == 0, 'left', 'right'
)
validation_predictions['right_probability'] = validation_result['right_probability']
validation_predictions['correct'] = (
    validation_result['predictions'] == validation_result['targets']
)
validation_predictions.to_csv(OUTPUT_ROOT / 'validation_predictions.csv', index=False)

participant_metrics = []
for participant, rows in validation_predictions.groupby('participant'):
    participant_metrics.append({
        'participant': participant,
        'trials': len(rows),
        'accuracy': accuracy_score(rows['label_id'], rows['predicted_label_id']),
        'balanced_accuracy': balanced_accuracy_score(
            rows['label_id'], rows['predicted_label_id']
        ),
    })
participant_metrics = pd.DataFrame(participant_metrics).sort_values(
    'balanced_accuracy', ascending=False
)
participant_metrics.to_csv(OUTPUT_ROOT / 'validation_per_participant_metrics.csv', index=False)
display(participant_metrics)
display(participant_metrics[['accuracy', 'balanced_accuracy']].describe())

TRAINED_MODEL_PATH = OUTPUT_ROOT / 'trained_motor_imagery_mlp.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': '54 -> 64 -> 32 -> 2',
    'feature_names': FEATURE_NAMES,
    'class_names': ['left', 'right'],
    'seed': SEED,
    'best_epoch': best_epoch,
    'trained': True,
    'tested': False,
}, TRAINED_MODEL_PATH)

print(f'Best epoch: {best_epoch}')
print(f'Training duration: {time.perf_counter() - started:.1f} seconds')
print(f'Trained checkpoint: {TRAINED_MODEL_PATH}')
print('Test split evaluated: NO')

## Current status

The architecture is created in the earlier cells. If the training cell has been executed, the fitted checkpoint and training/validation reports are stored under `data/processed/neural_net`. The test split remains untouched.